In [10]:
# https://github.com/arpanghosh8453/programs/blob/master/Fitbit%20Data%20Analyzer/Fitbit%20HR%20analyzer.ipynb
# https://dev.fitbit.com/
# https://dev.fitbit.com/build/reference/web-api/basics/
# Implicit Grant Flow
# You need the following modules, if you don't have them, use pip install <module-name>

import requests
import pandas as pd
import datetime
import numpy as np
import json

import dash
import dash_core_components as dcc
import dash_html_components as html
import plotly.express as px
import jupyter_dash
from dash.dependencies import Input, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

In [11]:
with open("data/credentials.json", "r") as file:
    credentials = json.load(file)
    fitbit_cr = credentials['fitbit_cr']
    fitbit_key = fitbit_cr['KEY']
    username = fitbit_cr['USERNAME']

In [12]:
# day_range_length = 1

# start_date = str((datetime.datetime.now() - datetime.timedelta(days=day_range_length)).date())
# #start_date = "2020-01-01"

# end_date = str(datetime.datetime.date(datetime.datetime.now())) 
# #end_date = "2020-01-02"

### Date range

In [13]:
start_date = '2020-01-01'
end_date = '2020-01-08'
print(start_date, "to", end_date)

2020-01-01 to 2020-01-08


In [14]:
#Update your start and end dates here in yyyy-mm-dd format 
start = datetime.datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.datetime.strptime(end_date, "%Y-%m-%d")

date_array = (start + datetime.timedelta(days=x) for x in range(0, (end-start).days))

day_list = []
for date_object in date_array:
    day_list.append(date_object.strftime("%Y-%m-%d"))
    
print("day range : ", day_list)

day range :  ['2020-01-01', '2020-01-02', '2020-01-03', '2020-01-04', '2020-01-05', '2020-01-06', '2020-01-07']


In [16]:
df_all = pd.DataFrame()

### Heart Rate requests

In [18]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

for single_day in day_list:
    response = requests.get("https://api.fitbit.com/1/user/-/activities/heart/date/"+ single_day +"/1d/1min/time/00:00/23:59.json", headers=header).json()
    try:
        df = pd.DataFrame(response['activities-heart-intraday']['dataset'])
        date = pd.Timestamp(single_day).strftime('%Y-%m-%d')
        df = df.set_index(pd.to_datetime(date + ' ' + df['time'].astype(str)))
        #print(df)
        df_all = df_all.append(df, sort=True)
    except KeyError as e:
        print("No data for the given date", date)
    
# df_all.index.set_names('dateTime', inplace = True)   
# del df_all['time']
df_all

In [20]:
del df_all['time']

In [21]:
df_all

,value
time,
2020-01-01 00:00:00,57
2020-01-01 00:01:00,57
2020-01-01 00:02:00,56
2020-01-01 00:03:00,56
2020-01-01 00:04:00,57
...,...
2020-01-07 23:55:00,72
2020-01-07 23:56:00,76
2020-01-07 23:57:00,75


In [13]:
# #Put the interval you want to take the average of the imported data from fitbit with 2-5 sec interval. Default 10 minute
# summary_df = (df_all['value'].resample('10Min').mean())
# summary_df

In [55]:
#Put the interval you want to take the average of the imported data from fitbit with 2-5 sec interval. Default 10 minute
summary_df = df_all
summary_df

,value,time
time,,
2020-01-01 00:00:00,57,2020-01-01 00:00:00
2020-01-01 00:01:00,57,2020-01-01 00:01:00
2020-01-01 00:02:00,56,2020-01-01 00:02:00
2020-01-01 00:03:00,56,2020-01-01 00:03:00
2020-01-01 00:04:00,57,2020-01-01 00:04:00
...,...,...
2020-01-07 23:55:00,72,2020-01-07 23:55:00
2020-01-07 23:56:00,76,2020-01-07 23:56:00
2020-01-07 23:57:00,75,2020-01-07 23:57:00


In [143]:
# trying to create empty <None> list so that gaps show up  in the heart rate dashboard when I don't have it on
blank_range = pd.DataFrame()
blank_range['time'] = pd.date_range("1/1/2020", "1/8/2020", freq="T")
#blank_range['time'].to_string()
blank_range['value'] = None
blank_range

,time,value
0,2020-01-01 00:00:00,None
1,2020-01-01 00:01:00,None
2,2020-01-01 00:02:00,None
3,2020-01-01 00:03:00,None
4,2020-01-01 00:04:00,None
...,...,...
10076,2020-01-07 23:56:00,None
10077,2020-01-07 23:57:00,None
10078,2020-01-07 23:58:00,None
10079,2020-01-07 23:59:00,None


In [144]:
df2 = pd.read_csv('~/qs/fitbit/data/hr.csv')
df2

,value,time
0,57,2020-01-01 00:00:00
1,57,2020-01-01 00:01:00
2,56,2020-01-01 00:02:00
3,56,2020-01-01 00:03:00
4,57,2020-01-01 00:04:00
...,...,...
9658,72,2020-01-07 23:55:00
9659,76,2020-01-07 23:56:00
9660,75,2020-01-07 23:57:00
9661,76,2020-01-07 23:58:00


In [145]:
df2['time'] = pd.to_datetime(df2['time'])
type(df2['time'][0])

pandas._libs.tslibs.timestamps.Timestamp

In [146]:
new_df = pd.merge(blank_range, df2, left_on=['time','value'], right_on = ['time','value'],how='left')
new_df

ValueError: You are trying to merge on object and int64 columns. If you wish to proceed you should use pd.concat

In [142]:
from IPython.display import display
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(blank_range)

                     time value
0     2020-01-01 00:00:00  None
1     2020-01-01 00:01:00  None
2     2020-01-01 00:02:00  None
3     2020-01-01 00:03:00  None
4     2020-01-01 00:04:00  None
5     2020-01-01 00:05:00  None
6     2020-01-01 00:06:00  None
7     2020-01-01 00:07:00  None
8     2020-01-01 00:08:00  None
9     2020-01-01 00:09:00  None
10    2020-01-01 00:10:00  None
11    2020-01-01 00:11:00  None
12    2020-01-01 00:12:00  None
13    2020-01-01 00:13:00  None
14    2020-01-01 00:14:00  None
15    2020-01-01 00:15:00  None
16    2020-01-01 00:16:00  None
17    2020-01-01 00:17:00  None
18    2020-01-01 00:18:00  None
19    2020-01-01 00:19:00  None
20    2020-01-01 00:20:00  None
21    2020-01-01 00:21:00  None
22    2020-01-01 00:22:00  None
23    2020-01-01 00:23:00  None
24    2020-01-01 00:24:00  None
25    2020-01-01 00:25:00  None
26    2020-01-01 00:26:00  None
27    2020-01-01 00:27:00  None
28    2020-01-01 00:28:00  None
29    2020-01-01 00:29:00  None
30    20

In [22]:
# import matplotlib.pyplot as plt

# plt.rcParams["figure.figsize"]=20,8

# # Heart rate data summary [10min avg] from start date[2021-03-18] to end date[2021-03-21] 
# #if you are using matplotlib directly in python ( py file ) then use plt.plot(summary_df,kind='area')
# summary_df.plot.area(ylim=(40,160))

In [37]:
summary_df.to_csv('data/hr.csv', index=None, encoding='utf-8')

In [36]:
summary_df['time'] = summary_df.index
summary_df

,value,time
time,,
2020-01-01 00:00:00,57,2020-01-01 00:00:00
2020-01-01 00:01:00,57,2020-01-01 00:01:00
2020-01-01 00:02:00,56,2020-01-01 00:02:00
2020-01-01 00:03:00,56,2020-01-01 00:03:00
2020-01-01 00:04:00,57,2020-01-01 00:04:00
...,...,...
2020-01-07 23:55:00,72,2020-01-07 23:55:00
2020-01-07 23:56:00,76,2020-01-07 23:56:00
2020-01-07 23:57:00,75,2020-01-07 23:57:00


### Resting Heart Rate requests

In [17]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

response = requests.get("https://api.fitbit.com/1/user/-/activities/heart/date/"+start_date+"/"+end_date+".json", headers=header).json()

In [24]:
#response

In [25]:
#all_resting_HR_list = []
for i in response['activities-heart']:
    try:
        resting_dict = { 'dateTime':i['dateTime'], "resting_HR":i['value']['restingHeartRate']}
        all_resting_HR_list.append(resting_dict)
    except KeyError as e:
        print("No data for the given date", i['dateTime'])
    
resting_HR_df = pd.DataFrame(all_resting_HR_list)
resting_HR_df['time'] = resting_HR_df.dateTime.apply(pd.Timestamp)
 
# resting_HR_df.set_index("dateTime", inplace = True)
# resting_HR_df['time'] = resting_HR_df.index

In [27]:
del resting_HR_df['dateTime']
resting_HR_df

,resting_HR,time
0,56,2020-01-01
1,58,2020-01-02
2,57,2020-01-03
3,58,2020-01-04
4,57,2020-01-05
5,58,2020-01-06
6,59,2020-01-07
7,58,2020-01-08
8,56,2020-01-01
9,58,2020-01-02


### Step Count

In [28]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

response = requests.get("https://api.fitbit.com/1/user/-/activities/steps/date/"+start_date+"/"+end_date+"/1min.json", headers=header).json()['activities-steps']

In [29]:
step_df = pd.DataFrame(response)
step_df['time'] = pd.to_datetime(step_df['dateTime'].apply(pd.Timestamp)).dt.date
del step_df['dateTime']
step_df['Step Count'] = step_df['value'].apply(int)
del step_df['value']
step_df

,dateTime,time,Step Count
0,2020-01-01,2020-01-01,1041
1,2020-01-02,2020-01-02,321
2,2020-01-03,2020-01-03,1659
3,2020-01-04,2020-01-04,8407
4,2020-01-05,2020-01-05,4520
5,2020-01-06,2020-01-06,3202
6,2020-01-07,2020-01-07,1542
7,2020-01-08,2020-01-08,4072


### Distance

In [39]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

response = requests.get("https://api.fitbit.com/1/user/-/activities/distance/date/"+start_date+"/"+end_date+"/1min.json", headers=header).json()['activities-distance']

In [40]:
distance_df = pd.DataFrame(response)
distance_df['time'] = pd.to_datetime(distance_df['dateTime'].apply(pd.Timestamp)).dt.date
del distance_df['dateTime']
distance_df['Distance in KM'] = distance_df['value'].apply(float)
del distance_df['value']
distance_df

,time,Distance in KM
0,2020-01-01,0.82576
1,2020-01-02,0.24246
2,2020-01-03,1.25603
3,2020-01-04,6.37055
4,2020-01-05,3.49040
5,2020-01-06,2.31321
6,2020-01-07,1.16396
7,2020-01-08,3.08456
